# V2 Phase 9 — Colab GPU Multi-Agent RAG smoke

**Before running:** Runtime → Change runtime type → **GPU** (T4 or better).

This notebook:
1. Clones V2 from GitHub
2. Installs dependencies
3. Restores the Phase 6 KB from Google Drive (or rebuilds if missing)
4. Runs index preflight
5. Runs Phase 9 Multi-Agent RAG smoke (`llama_cpp`, n=3)
6. Saves results to Drive

## Setup

Push latest V2 (including Phase 9) to branch `cursor/empty-v2-workspace`, then run all cells.

Requires Phase 8 KB on Drive at `MyDrive/MSc-RAG/artifacts/knowledge_base/` (from `colab_phase8_smoke.ipynb` section 7). If missing, cell 3 falls back to `build_index.py`.

**Outputs:** `results/config/phase9_smoke_test.json`, `phase9_multi_agent_smoke.json`

## 1. Clone GitHub repo and enter V2

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/syedsafiullah777/CAPSTONE--RAG-WITH-UNCERTAINITY-QUANTIFICATION-.git'
BRANCH = 'cursor/empty-v2-workspace'
CLONE_DIR = Path('/content/capstone-rag')

if CLONE_DIR.exists():
    !rm -rf {CLONE_DIR}

print('Cloning branch:', BRANCH)
result = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(CLONE_DIR)],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'git clone failed. Push V2/ to GitHub on branch {BRANCH!r} first.')

V2_ROOT = CLONE_DIR / 'V2'
if not (V2_ROOT / 'scripts' / 'smoke_multi_agent.py').is_file():
    raise FileNotFoundError(f'Phase 9 script missing at {V2_ROOT}. Push Phase 9 to GitHub first.')

os.chdir(V2_ROOT)
sys.path.insert(0, str(V2_ROOT))
print('OK — working in V2_ROOT:', V2_ROOT)
!git -C {CLONE_DIR} log -1 --oneline

## 2. Install dependencies

In [ ]:
!pip -q install -r requirements.txt
!pip -q install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

## 3. Restore knowledge base from Drive (or rebuild)

Does **not** copy the Mac Chroma database. Reuses the Colab-built index from Phase 8 when available.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

V2 = Path('/content/capstone-rag/V2')
DRIVE_ROOT = Path('/content/drive/MyDrive/MSc-RAG')
restored = True

for rel in ('artifacts/knowledge_base/index', 'artifacts/knowledge_base/documents'):
    src = DRIVE_ROOT / rel
    dst = V2 / 'knowledge_base' / rel.split('/')[-1]
    if not src.is_dir():
        print('Missing on Drive:', src)
        restored = False
        break
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print('restored', dst)

if not restored:
    print('Drive KB not found — falling back to build_index.py (Option B)...')
    !PYTHONPATH=. python scripts/build_index.py --distractors 50

## 4. Index preflight validation

In [ ]:
!PYTHONPATH=. python scripts/validate_kb_index.py

## 5. Phase 9 Multi-Agent RAG smoke (llama_cpp, n=3)

In [ ]:
!PYTHONPATH=. python scripts/smoke_multi_agent.py --backend llama_cpp --limit 3

## 6. Check results

In [ ]:
import json
from pathlib import Path

fp = Path('results/config/phase9_runtime_fingerprint.json')
smoke = Path('results/config/phase9_smoke_test.json')
detail = Path('results/config/phase9_multi_agent_smoke.json')
print('fingerprint:', fp.is_file())
print('smoke_test:', smoke.is_file())
print('detail:', detail.is_file())
if smoke.is_file():
    data = json.loads(smoke.read_text())
    print('status:', data.get('status'))
    print('actual:', data.get('actual'))
if detail.is_file():
    detail_data = json.loads(detail.read_text())
    print('backend:', detail_data.get('backend'))
    for case in detail_data.get('cases', []):
        vr = case.get('verification_result') or {}
        print(
            case.get('question_id'),
            'n_evidence=',
            len(case.get('retrieved_evidence') or []),
            'verify=',
            vr.get('verification_score'),
            'status=',
            vr.get('status'),
            'answer_len=',
            len(case.get('answer') or ''),
        )

## 7. Save Phase 9 results to Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive', force_remount=True)
V2 = Path('/content/capstone-rag/V2')
dest = Path('/content/drive/MyDrive/MSc-RAG/configs/phase9')
dest.mkdir(parents=True, exist_ok=True)
for name in (
    'phase9_runtime_fingerprint.json',
    'phase9_smoke_test.json',
    'phase9_multi_agent_smoke.json',
    'phase9_multi_agent_smoke.jsonl',
):
    src = V2 / 'results' / 'config' / name
    if src.is_file():
        shutil.copy2(src, dest / name)
        print('copied', name)
print('Done:', dest)